In [1]:
import sys
import os
sys.executable

'/N/u/echimal/Quartz/.conda/envs/integration_env/bin/python'

In [2]:
import pandas as pd
import numpy as np
import scanpy as sc
import anndata
import matplotlib.pyplot as plt

In [2]:
#Data path
data_path = '/N/u/echimal/Quartz/Desktop/CLR_MRI/Human_GeoMx_Sep2025/'

In [4]:
def wta_to_AnnData(data_path, target_counts_file, probe_counts_file, feature_annotations_file,
                   sample_annotations_file, saving = 'AnnData', returning = False):

    target_counts_path = data_path + target_counts_file
    probe_counts_path = data_path + probe_counts_file
    feature_annotations_path = data_path + feature_annotations_file
    sample_annotations_path = data_path + sample_annotations_file
    
    target_counts = pd.read_csv(target_counts_path, index_col=0)
    probe_counts = pd.read_csv(probe_counts_path, index_col=0)
    feature_annotations = pd.read_csv(feature_annotations_path)
    sample_annotations = pd.read_csv(sample_annotations_path)
    
    sample_annotations = sample_annotations.loc[:, ~sample_annotations.columns.str.contains("^Unnamed")]
    target_counts = target_counts.loc[:, ~target_counts.columns.str.contains("^Unnamed")]
    probe_counts = probe_counts.loc[:, ~probe_counts.columns.str.contains("^Unnamed")]
    feature_annotations['Negative'] = feature_annotations['Negative'].astype(str).str.upper().map({'TRUE': True, 'FALSE': False})

    
    sample_annotations.index = sample_annotations['ROI']
    sample_annotations = sample_annotations.loc[target_counts.columns,:]
    
    negative_probe_names = np.array(feature_annotations['ProbeID'].loc[feature_annotations['Negative']])
    negative_probe_counts = probe_counts.loc[negative_probe_names,:]
    negative_probe_counts = negative_probe_counts.loc[:,target_counts.columns]
    
    feature_annotations.index = feature_annotations['TargetName']
    feature_annotations = feature_annotations[~feature_annotations['Negative']]
    target_counts = target_counts.loc[feature_annotations.index,:]
    
    adata = sc.AnnData(X = np.array(target_counts).T)
    adata.var_names = target_counts.index
    adata.var = feature_annotations
    adata.obs = sample_annotations
    adata.obsm['negProbes'] = np.array(negative_probe_counts.T)
    
    adata.obs.index = range(len(adata.obs.index))
    adata.var.index = range(len(adata.var.index))
    
    adata.var_names = np.array(adata.var['TargetName'])
    
    if saving:
        adata.write(data_path + saving)
    
    if returning:
        return adata

In [5]:
wta_to_AnnData(data_path = data_path,
               target_counts_file = 'CAA-AD_Raw_TargetCountMatrix.csv',
                probe_counts_file = 'CAA-AD_Raw_BioProbeCountMatrix.csv',
                feature_annotations_file = 'CAA-AD_Feature_Annotations.csv',
                sample_annotations_file = 'CAA-AD_Sample_Annotations.csv',
              saving = 'CAA-AD_AnnData.h5ad',
              returning = False)

In [11]:
import os
import sys
import cell2location
data_path = '/N/u/echimal/Quartz/Desktop/CLR_MRI/Human_GeoMx_Sep2025/'

os.chdir(data_path)

print("Working directory:", os.getcwd())
print("cell2location version:", cell2location.__version__)

Working directory: /N/project/CLR_MRI/Human_GeoMx_Sep2025
cell2location version: 0.1.5


In [12]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


False
CPU only


In [16]:
sys.path.append('/N/project/CLR_MRI/Human_GeoMx_Sep2025/SpaceJam')

import scanpy as sc
import numpy as np
import pandas as pd
from re import sub
from spacejam.models import LocationModelWTAMultiExperimentHierarchicalGeneLevel
import cell2location
import os

#WTA Anndata object 
working_dir = '/N/u/echimal/Quartz/Desktop/CLR_MRI/Human_GeoMx_Sep2025/'
adata_wta = sc.read_h5ad(f"{working_dir}CAA-AD_AnnData.h5ad")

In [19]:
#Split the data by "disease_status"
adata_wta_ADCAA = adata_wta[adata_wta.obs["disease_status"] == "AD-CAA"].copy()
adata_wta_Control = adata_wta[adata_wta.obs["disease_status"] == "Control"].copy()
